# SWEAT tutorials: STIC processing

Explain how to estimate evapotranspiration with STIC model on a raster

In [ ]:
from __future__ import annotations

In [ ]:
%load_ext autoreload
%autoreload 2

## Imports

In [ ]:
import os
from pprint import pprint

In [ ]:
from sweat.api import read_input_data, run_stic
from sweat.stic.config import check_config_stic

In [ ]:
import sweat.extra.plot_notebook as pltn

In [ ]:
from extra.process import remove_directory, tree

In [ ]:
import logging

from sweat.logging import LoggerManager

LoggerManager.set_level(logging.INFO)

## Case: Simple execution

We will first run STIC with a simple configuration.

#### Input configuration

For simple configuration, the only required sections are:

  * `input`: to specify the input data
  * `output`: to specify where the results are written

In [ ]:
!cat data/stic_input.json

#### Execution with command line

Clean output directory

In [ ]:
remove_directory("out")

Run the command line

In [ ]:
!stic data/stic_input.json

Look at the output directory

In [ ]:
tree("out")

#### Execution with high-level API

Input configuration 

In [ ]:
config_dict = {
    "input": {
        "path": os.path.join(
            "data", "data_test_20230303T112730.tif"
        ),
        "date": "2023-03-03T11:27:30-00:00",
    },
    "output": {"path": "out"},
    "params": {},
}

Check input configuration

In [ ]:
config = check_config_stic(config_dict)

In [ ]:
pprint(config, sort_dicts=False)

*You can see that parameters have been added to the configuration. The code supplements the configuration with default parameters that had not been specified.*

Read

In [ ]:
input_data = read_input_data(config_dict["input"])

In [ ]:
pltn.plot_dataset(input_data)

Run 

In [ ]:
inst_xr, daily_xr, ws_xr = run_stic(input_data, config["params"])

You get tree datatsets:

* `inst_xr`: Instatenous data
* `daily_xr`: Daily data
* `ws_xr`: Water stress data


In [ ]:
inst_xr

In [ ]:
daily_xr

In [ ]:
ws_xr

Visualize results

In [ ]:
pltn.plot_dataset(inst_xr)

In [ ]:
pltn.plot_dataset(daily_xr)

In [ ]:
pltn.plot_dataset(ws_xr)

## Case: Detailed configuration

#### Configuration

In this example, we will use a slightly more complex configuration, particularly regarding the input settings. You can note how to configure the filtering section to specify which pixels are valid.

You can find all the available settings [here](https://github.com/CNES/sweat/blob/main/docs/stic/configuration.md).



In [ ]:
config_dict = {
    "input": {
        "path": os.path.join(
            "data", "data_test_20230303T112730.tif"
        ),
        "date": "2023-03-03T11:27:30-00:00",
    },
    "output": {"path": "out"},
    "params": {
        "prepare": {"use_topo": True, "selected_radiation": "msg"},
        "filtering": {
            "tdp": {"op": ">=", "value": -30.0},
            "cloud": {"op": "==", "value": 0},  # keep clear pixels
            "water": {"op": "==", "value": 0},  # keep non water pixels
            "qa": {"op": "==", "value": 1},  # keep pixels with qa = 1
            "ndvi": {
                "and": [{"op": ">=", "value": 0}, {"op": "<=", "value": 1}]
            },  # Keep NDVI between [0,1]
            "fcover": {
                "and": [{"op": ">=", "value": 0}, {"op": "<=", "value": 1}]
            },  # Keep Fcover between [0,1]
            "albedo": {
                "and": [{"op": ">=", "value": 0}, {"op": "<=", "value": 1}]
            },  # Keep Fcover between [0,1]
            "height": {
                "and": [{"op": ">=", "value": 0}, {"op": "<=", "value": 300}]
            },  # Keep height between [0,300]
        },
        "stic": {"version": "1.4", "threshold": 0.01, "nb_steps": 15},
        "daily": {"method": "toa", "use_topo": True},
        "waterstress": {"method": "ef"},
    },
}

In [ ]:
pprint(config_dict, sort_dicts=False)

#### Execution

Check input file

In [ ]:
config = check_config_stic(config_dict)

In [ ]:
config

Read

In [ ]:
input_data = read_input_data(config_dict["input"])

Run

In [ ]:
inst_xr, daily_xr, ws_xr = run_stic(input_data, config["params"])

Visualize results

In [ ]:
pltn.plot_dataset(inst_xr)

In [ ]:
pltn.plot_dataset(daily_xr)

In [ ]:
pltn.plot_dataset(ws_xr)

## Case: detailed pipeline

Here, we will delve a bit deeper into the API and execute the pipeline steps one by one.

Additional imports

In [ ]:
from sweat.common import daily, filter
from sweat.common import waterstress as ws
from sweat.stic import main
from sweat.stic.config import STICParamsConfig
from sweat.stic.registry import MODEL_REGISTRY

Configuration

In [ ]:
config_dict = {
    "input": {
        "path": os.path.join(
            "data", "data_test_20230303T112730.tif"
        ),
        "date": "2023-03-03T11:27:30-00:00",
    },
    "output": {"path": "out"},
    "params": {
        "prepare": {"use_topo": True, "selected_radiation": "msg"},
        "filtering": {
            "tdp": {"op": ">=", "value": -30.0},
            "cloud": {"op": "==", "value": 0},  # keep clear pixels
            "water": {"op": "==", "value": 0},  # keep non water pixels
            "qa": {"op": "==", "value": 1},  # keep pixels with qa = 1
            "ndvi": {
                "and": [{"op": ">=", "value": 0}, {"op": "<=", "value": 1}]
            },  # Keep NDVI between [0,1]
            "fcover": {
                "and": [{"op": ">=", "value": 0}, {"op": "<=", "value": 1}]
            },  # Keep Fcover between [0,1]
            "albedo": {
                "and": [{"op": ">=", "value": 0}, {"op": "<=", "value": 1}]
            },  # Keep Fcover between [0,1]
            "height": {
                "and": [{"op": ">=", "value": 0}, {"op": "<=", "value": 300}]
            },  # Keep height between [0,300]
        },
        "stic": {"version": "1.4", "threshold": 0.01, "nb_steps": 15},
        "daily": {"method": "geo", "name": "msg"},
        "waterstress": {"method": "ef"},
    },
}

Check configuration

In [ ]:
config = check_config_stic(config_dict)
# Validate parameters config
params_config = STICParamsConfig.model_validate(config["params"])

In [ ]:
pprint(config)

Read input data

In [ ]:
data = read_input_data(config["input"])

In [ ]:
pltn.plot_dataset(data)

In [ ]:
pltn.plot_dem(data[["height", "slope", "aspect"]])

Prepare data

The following steps are performed:

- Compute LST in celsius
- Compute relative humidity
- Compute local time
- Mask pixel if nan in input data
- Create flags mask
- Compute vegetation indices required by the model if necessary

In [ ]:
data = main.prepare(data, **params_config.prepare.model_dump())

In [ ]:
pltn.plot_dataset(
    data[["lst", "ta", "tdp", "rh", "fcover", "local_time", "rn", "ln"]]
)

Filter data

- Mask NaN values
- Identify valid pixels

In [ ]:
valid_mask, flags_mask = filter.find_valid_pixels(
    data,
    nan_config=MODEL_REGISTRY[params_config.stic.version].inputs,
    valid_config=params_config.filtering.model_dump(by_alias=True),
)
data["valid"] = valid_mask
data["flags"] = flags_mask

In [ ]:
if (data["valid"].data == 0).sum() == 0:
    print("No invalid pixels")
else:
    pltn.plot_mask(data=data, mask="valid", invert=True)

In [ ]:
pltn.plot_flags(data, "flags")

Compute instant LE

In this step, the STIC model is executed on all valid pixels.

In [ ]:
inst_xr = main.run(data, **params_config.stic.model_dump())

In [ ]:
inst_xr

In [ ]:
pltn.plot_dataset(inst_xr)

Extrapolate at daily scale

In [ ]:
daily_xr = daily.extrapolate_at_daily_scale(
    inst_xr,
    variables=["le", "et"],
    extra=data,
    **params_config.daily.model_dump(),
)

In [ ]:
pltn.plot_dataset(daily_xr)

Compute water stress indices

In [ ]:
ws_xr = ws.run(et=inst_xr, **params_config.waterstress.model_dump())

In [ ]:
ws_xr

In [ ]:
pltn.plot_dataset(ws_xr)